In [ ]:
## WEB SCRAPER TO retrieve to from inec website 

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Base URL pattern
base_url = "https://integrity.ng/index.php/units/browse/"
page = 1
data = []

while True:
    url = f"{base_url}{page}"
    print(f"🔍 Scraping page {page} -> {url}")
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"❌ Failed to load page {page}. Status code:", response.status_code)
        break

    soup = BeautifulSoup(response.content, "html.parser")
    rows = soup.select("table tbody tr")

    if not rows:
        print("✅ No more data found. Stopping.")
        break

    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 7:
            state = cols[0].text.strip()
            lga = cols[1].text.strip()
            ward = cols[2].text.strip()
            polling_station = cols[3].text.strip()
            code = cols[4].text.strip()
            agent_phone = cols[5].text.strip()
            pvc = cols[6].text.strip()
            data.append([state, lga, ward, polling_station, code, agent_phone, pvc])

    print(f"✅ Page {page} scraped successfully. Waiting 2 seconds before next page...")
    page += 1
    time.sleep(2)  # polite delay for the  website to load  

# Create DataFrame and save to CSV
df = pd.DataFrame(data, columns=["State", "LGA", "Ward", "Polling Station", "Code", "Agent Phone", "PVC"])
df.to_csv("polling_units_integrity_all_pages.csv", index=False)
print(f"🎉 Done! Saved polling_units_integrity_all_pages.csv with {len(df)} total entries.")


In [ ]:
## sorting  through csv data  and  grouping  all the  data  by  state 
import pandas as pd
import os

# === 1. Load dataset ===
file_path = "polling_units_integrity_all_pages.csv"
data = pd.read_csv(file_path)

# === 2. Define column names ===
state_col = "State"
lga_col = "LGA"
ward_col = "Wards"
pu_col = "Polling Station"  # Update this if different in your data

# === 3. Clean up text columns ===
for col in [state_col, lga_col, ward_col, pu_col]:
    data[col] = data[col].astype(str).str.strip()

# === 4. Output directory ===
output_dir = "OUTPUT_STATES"
os.makedirs(output_dir, exist_ok=True)

# === 5. Helper: clean file name ===
def clean_name(name):
    return "".join(c for c in name if c.isalnum() or c in (' ', '_', '-')).strip().replace(" ", "_")

# === 6. Group by State and write Excel ===
for state_name, state_df in data.groupby(state_col):
    cleaned_state = clean_name(state_name)
    output_path = os.path.join(output_dir, f"{cleaned_state}.xlsx")

    # --- Sheet 1: State, LGA ---
    sheet1_df = state_df[[state_col, lga_col]].drop_duplicates().sort_values(by=[state_col, lga_col])

    # --- Sheet 2: LGA, Ward ---
    sheet2_df = state_df[[lga_col, ward_col]].drop_duplicates().sort_values(by=[lga_col, ward_col])

    # --- Sheet 3: LGA, Ward, Polling Unit ---
    sheet3_df = state_df[[lga_col, ward_col, pu_col]].drop_duplicates().sort_values(by=[lga_col, ward_col, pu_col])

    # --- Write to Excel ---
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        sheet1_df.to_excel(writer, index=False, sheet_name='State_LGA')
        sheet2_df.to_excel(writer, index=False, sheet_name='LGA_Wards')
        sheet3_df.to_excel(writer, index=False, sheet_name='LGA_Wards_PUs')

    print(f"✅ Saved Excel file for state: {state_name} → {output_path}")

print("\n🎯 Done! One Excel file per state created with 3 sheets each.")
